In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# 设置保存路径
save_path = r"D:\CDT\04-结果\fig3_v3"

# 读取数据
file_path = r"D:\CDT\03-数据\CDT_v3.csv"
df = pd.read_csv(file_path)

# 定义urban systems列（保持原始顺序）
urban_systems = ['People', 'Household', 'Land_and_agriculture',
                 'Workplace', 'Transportation', 'IoT_technology']

# 确保列存在并处理数据
us_columns = [col for col in urban_systems if col in df.columns]

for col in us_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 确保Cluster列是数值类型
df['Cluster'] = pd.to_numeric(df['Cluster'], errors='coerce')

# 获取Cluster的排序：1,2,3,4,5,0
clusters = [1, 2, 3, 4, 5, 0]

# 定义颜色
cluster_colors = ['#ee8d87', '#fedfb2', '#A9CACE', '#feece7', '#fff9eb', '#deeeed']  # cluster1-5,0的颜色
not_involved_color = '#ececea'  # 灰色表示没涉及的部分

# ==================== 主图：fig3a.png ====================

# 计算比例数据 - 每个urban system单独计算百分比
percentage_data = {}

for us in us_columns:
    us_data = {}

    # 先计算有效数据总数（排除该urban system为NaN的数据）
    valid_data = df[df[us].notna()]
    total_valid_cases = len(valid_data)

    if total_valid_cases > 0:
        # 计算每个cluster的涉及情况
        for cluster in clusters:
            # 该cluster且涉及该us的情况（us=1）
            cluster_data = valid_data[valid_data['Cluster'] == cluster]
            involved_cases = len(cluster_data[cluster_data[us] == 1])

            # 计算该cluster涉及的百分比
            percentage_involved = (involved_cases / total_valid_cases) * 100
            us_data[f'Cluster{cluster}'] = percentage_involved

        # 计算没涉及该us的总情况（us=0）
        not_involved_cases = len(valid_data[valid_data[us] == 0])
        percentage_not_involved = (not_involved_cases / total_valid_cases) * 100
        us_data['Not_Involved'] = percentage_not_involved

    percentage_data[us] = us_data

# 转换为DataFrame
percentage_df = pd.DataFrame(percentage_data).T

# 堆叠顺序：cluster1,2,3,4,5,0，最后是Not_Involved
stack_order = ['Cluster1', 'Cluster2', 'Cluster3', 'Cluster4', 'Cluster5', 'Cluster0', 'Not_Involved']
stack_colors = cluster_colors + [not_involved_color]

# 创建横向堆叠条形图
fig, ax = plt.subplots(figsize=(12/2.54, 3/2.54))

# 设置背景色为白色
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# 设置条形位置（y轴）
y_pos = np.arange(len(us_columns))
bar_height = 0.7

# 绘制横向堆叠条形
left = np.zeros(len(us_columns))

for i, stack_type in enumerate(stack_order):
    if stack_type in percentage_df.columns:
        values = percentage_df[stack_type].values
        ax.barh(y_pos, values, bar_height, left=left,
                color=stack_colors[i], edgecolor='white', linewidth=1.5)
        left += values

# 移除所有文字标签
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('')

# 移除边框
for spine in ax.spines.values():
    spine.set_visible(False)

# 移除图例
if ax.get_legend():
    ax.get_legend().remove()

# 移除网格
ax.grid(False)

# 设置x轴范围为0-100%
ax.set_xlim(0, 100)

# 保存主图
plt.tight_layout()
plt.savefig(f"{save_path}/fig3a.png", dpi=300, bbox_inches='tight', pad_inches=0)
plt.close(fig)

# ==================== 标签图：fig3a_label.png ====================

# 合并颜色：前6个是cluster颜色，最后是not_involved
legend_colors = cluster_colors + [not_involved_color]

# 创建图形
fig_width = 0.5/2.54
fig_height = 3/2.54
fig = plt.figure(figsize=(fig_width, fig_height))
fig.patch.set_facecolor('white')

# 创建坐标轴
ax = fig.add_axes([0, 0, 1, 1])
ax.set_facecolor('white')

# 创建7个y位置（垂直排列7个小条形：6个cluster + 1个not_involved）
y_pos = np.arange(7)
bar_height = 0.6  # 调整高度

# 绘制7个条形，每个显示一种颜色
for i in range(7):
    ax.barh(y_pos[i], 100, bar_height, left=0,
            color=legend_colors[i], edgecolor='white', linewidth=1.5)

# 移除所有文字标签和边框
ax.set_xticks([])
ax.set_yticks([])
ax.set_xticklabels([])
ax.set_yticklabels([])
ax.set_xlabel('')
ax.set_ylabel('')
ax.set_title('')

for spine in ax.spines.values():
    spine.set_visible(False)

ax.grid(False)

# 设置范围
ax.set_xlim(0, 100)
ax.set_ylim(-0.5, 6.5)  # 调整为7个条形

# 保存标签图
plt.tight_layout()
plt.savefig(f"{save_path}/fig3a_legend.png", dpi=300, bbox_inches='tight', pad_inches=0)
plt.close(fig)

print(f"图片已保存至：{save_path}")
print("fig3a.png: 主图表")
print("fig3a_legend.png: 颜色标签图")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import chi2_contingency
from matplotlib.colors import LinearSegmentedColormap

# ==================== 基本设置 ====================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 5
plt.rcParams['font.weight'] = 'normal'

save_path = r"D:\CDT\04-结果\fig3_v3"
file_path = r"D:\CDT\03-数据\CDT_v3.csv"

df = pd.read_csv(file_path)

urban_systems = [
    'People', 'Household', 'Land_and_agriculture',
    'Workplace', 'Transportation', 'IoT_technology'
]
us_columns = [c for c in urban_systems if c in df.columns]

for col in us_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

n_us = len(us_columns)

# ==================== 1. 共现网络图 ====================
cooccurrence_matrix = np.zeros((n_us, n_us), dtype=int)

for _, row in df.iterrows():
    idxs = [i for i, us in enumerate(us_columns) if row.get(us) == 1]
    for i in idxs:
        for j in idxs:
            if i < j:
                cooccurrence_matrix[i, j] += 1

fig1, ax1 = plt.subplots(figsize=(4/2.54, 4/2.54), dpi=600)
fig1.subplots_adjust(0, 0, 1, 1)

angles = np.linspace(0, 2*np.pi, n_us, endpoint=False)
positions = [(np.cos(a), np.sin(a)) for a in angles]

min_w = cooccurrence_matrix.min()
max_w = cooccurrence_matrix.max()

for i in range(n_us):
    for j in range(i+1, n_us):
        w = cooccurrence_matrix[i, j]
        if w > 0:
            x1, y1 = positions[i]
            x2, y2 = positions[j]
            ax1.plot(
                [x1, x2], [y1, y2],
                color='#68a7be',
                linewidth=3.5 * (w - min_w) / (max_w - min_w),
                alpha=0.7 * (w - min_w) / (max_w - min_w),
                solid_capstyle='round'
            )

for i in range(n_us):
    x, y = positions[i]
    size = 200 + 600 * cooccurrence_matrix[i].sum() / cooccurrence_matrix.sum()
    ax1.add_patch(
        mpatches.Circle(
            (x, y), size/3000,
            facecolor='#ee7e77',
            edgecolor='#555555',
            linewidth=0.8,
            zorder=3
        )
    )

ax1.set_aspect('equal')
ax1.axis('off')
plt.savefig(f"{save_path}/fig3b.png", dpi=600, bbox_inches='tight')
plt.close()

# ==================== 2. Phi 系数热力图 ====================
phi = np.full((n_us, n_us), np.nan)
pval = np.full((n_us, n_us), np.nan)

for i in range(n_us):
    for j in range(i, n_us):
        if i == j:
            phi[i, j] = 1
            pval[i, j] = 0
        else:
            tab = pd.crosstab(df[us_columns[i]], df[us_columns[j]])
            if tab.shape == (2, 2):
                a, b, c, d = tab.values.flatten()
                phi_ij = (a*d - b*c) / np.sqrt((a+b)*(c+d)*(a+c)*(b+d))
                phi[i, j] = phi[j, i] = phi_ij
                _, p, _, _ = chi2_contingency(tab)
                pval[i, j] = pval[j, i] = p

valid_phi = phi[~np.isnan(phi) & (phi != 1)]
vmax = np.max(np.abs(valid_phi))
vmin = -vmax

cmap = LinearSegmentedColormap.from_list(
    'phi', ['#68a7be', '#ffffff', '#ee7e77']
)

fig2, ax2 = plt.subplots(figsize=(3/2.54, 3/2.54), dpi=600)
fig2.subplots_adjust(0, 0, 1, 1)

square = 0.8

for i in range(n_us):
    for j in range(i):
        val = phi[i, j]
        if np.isnan(val):
            continue

        x = j
        y = n_us - 1 - i
        color = cmap((val - vmin) / (vmax - vmin))

        ax2.add_patch(
            mpatches.Rectangle(
                (x - square/2, y - square/2),
                square, square,
                facecolor=color,
                edgecolor='none'
            )
        )

        ax2.text(x, y + 0.05, f"{val:.2f}",
                 ha='center', va='center', fontsize=5)

        if pval[i, j] < 0.001:
            s = "***"
        elif pval[i, j] < 0.01:
            s = "**"
        elif pval[i, j] < 0.05:
            s = "*"
        else:
            s = ""

        if s:
            ax2.text(x, y - 0.25, s,
                     ha='center', va='center', fontsize=5)

ax2.set_xlim(-0.5, n_us - 0.5)
ax2.set_ylim(-0.5, n_us - 0.5)
ax2.set_aspect('equal')
ax2.axis('off')

plt.savefig(f"{save_path}/fig3c.png", dpi=600, bbox_inches='tight')
plt.close()

# ==================== 3. 颜色条 ====================
fig_c, ax_c = plt.subplots(figsize=(0.6/2.54, 3/2.54), dpi=600)
fig_c.subplots_adjust(left=0.3, right=0.7, top=0.98, bottom=0.02)

sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])

cbar = fig_c.colorbar(sm, cax=ax_c, orientation='vertical')
cbar.ax.tick_params(labelsize=5, width=0.25, length=2)

for spine in ax_c.spines.values():
    spine.set_linewidth(0)

plt.savefig(f"{save_path}/fig3c_legend.png", dpi=600, bbox_inches='tight')
plt.close()

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter

# 设置显示选项
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

# 文件路径
file_path = r"D:\CDT\03-数据\CDT_v3.csv"
output_path = r"D:\CDT\04-结果\SDG分析_v3"

# 读取数据
df = pd.read_csv(file_path)

# 定义urban systems列
urban_systems = ['People', 'Household', 'Land_and_agriculture',
                 'Workplace', 'Transportation', 'IoT_technology']
us_columns = [col for col in urban_systems if col in df.columns]

# ==================== 数据预处理 ====================
def parse_sdgs_robust(cell):

    """解析Covered_SDGs列"""
    if pd.isna(cell):
        return []

    cell_str = str(cell).strip()
    if not cell_str:
        return []

    parts = cell_str.split(',')
    sdg_list = []

    for part in parts:
        part = part.strip()
        if part:
            try:
                sdg_num = int(part)
                if 1 <= sdg_num <= 17:
                    sdg_list.append(sdg_num)
            except ValueError:
                pass

    return sorted(list(set(sdg_list)))

# 解析SDG数据
df['SDG_list'] = df['Covered_SDGs'].apply(parse_sdgs_robust)

# 清理SDG Layer数据
layer_mapping = {'High': 'High', 'Moderate': 'Moderate', 'Low': 'Low'}
df['SDG_layer_clean'] = df['2025 SDG layer'].map(layer_mapping)

# 创建SDG存在矩阵
sdg_numbers = list(range(1, 18))
sdg_matrix = np.zeros((len(df), len(sdg_numbers)), dtype=int)

for idx, sdg_list in enumerate(df['SDG_list']):
    for sdg in sdg_list:
        if 1 <= sdg <= 17:
            sdg_matrix[idx, sdg-1] = 1

sdg_matrix_df = pd.DataFrame(sdg_matrix, columns=[f'SDG_{i}' for i in sdg_numbers])
df['SDG_count'] = sdg_matrix_df.sum(axis=1)

# ==================== Sheet 1: 单个SDG统计分析 ====================
sdg_analysis_data = []

for sdg_num in sdg_numbers:
    col_name = f'SDG_{sdg_num}'

    # 筛选包含该SDG的样本
    sdg_mask = sdg_matrix_df[col_name] == 1
    sdg_cases = df[sdg_mask]
    count = len(sdg_cases)

    # 计算频率（基于总样本数）
    frequency = count / len(df) * 100 if len(df) > 0 else 0

    # 计算Layer比例
    if count > 0:
        layer_dist = sdg_cases['SDG_layer_clean'].value_counts(normalize=True) * 100
        high_pct = layer_dist.get('High', 0)
        moderate_pct = layer_dist.get('Moderate', 0)
        low_pct = layer_dist.get('Low', 0)
    else:
        high_pct = moderate_pct = low_pct = 0

    # 计算Index Score统计
    sdg_scores = sdg_cases['2025 SDG Index Score'].dropna()
    if len(sdg_scores) > 0:
        score_mean = sdg_scores.mean()
        score_median = sdg_scores.median()
        score_count = len(sdg_scores)
    else:
        score_mean = score_median = np.nan
        score_count = 0

    sdg_analysis_data.append({
        'SDG编号': sdg_num,
        'SDG名称': f'SDG {sdg_num}',
        '出现数量': count,
        '频率(%)': round(frequency, 2),
        'High比例(%)': round(high_pct, 2),
        'Moderate比例(%)': round(moderate_pct, 2),
        'Low比例(%)': round(low_pct, 2),
        'IndexScore有效样本数': score_count,
        'IndexScore均值': round(score_mean, 2) if not np.isnan(score_mean) else 'N/A',
        'IndexScore中位数': round(score_median, 2) if not np.isnan(score_median) else 'N/A'
    })

# 创建DataFrame并按SDG编号排序
sdg_analysis_df = pd.DataFrame(sdg_analysis_data).sort_values('SDG编号')

# ==================== Sheet 2: SDG数量分布统计 ====================
# 统计每个SDG数量（包含0个的情况）
sdg_count_dist = df['SDG_count'].value_counts().sort_index()

sdg_count_data = []
for count_val in range(0, int(df['SDG_count'].max()) + 1):
    if count_val in sdg_count_dist.index:
        freq = sdg_count_dist[count_val]
    else:
        freq = 0

    percentage = freq / len(df) * 100

    sdg_count_data.append({
        'SDG数量': count_val,
        '样本数': freq,
        '比例(%)': round(percentage, 2)
    })

sdg_count_df = pd.DataFrame(sdg_count_data)

# ==================== Sheet 3: SDG组合统计分析 ====================
# 统计所有SDG组合（仅包含≥1个SDG的样本）
combo_counts = Counter()
combo_stats = {}

for idx, row in df.iterrows():
    sdg_list = row['SDG_list']
    layer = row['SDG_layer_clean']
    score = row['2025 SDG Index Score']

    # 跳过0个SDG的样本
    if len(sdg_list) == 0:
        continue

    # 创建排序后的组合字符串（确保SDG按编号排序）
    sorted_sdgs = sorted(sdg_list)
    combo_key = tuple(sorted_sdgs)
    combo_str = ', '.join([f"SDG{sdg}" for sdg in sorted_sdgs])

    # 更新组合计数
    combo_counts[combo_key] += 1

    # 初始化组合统计
    if combo_key not in combo_stats:
        combo_stats[combo_key] = {
            'combo_str': combo_str,
            'layer_counts': {'High': 0, 'Moderate': 0, 'Low': 0},
            'scores': []
        }

    # 更新layer计数
    if layer in combo_stats[combo_key]['layer_counts']:
        combo_stats[combo_key]['layer_counts'][layer] += 1

    # 收集Index Score
    if pd.notna(score):
        combo_stats[combo_key]['scores'].append(score)

# 处理组合数据
combo_analysis_data = []
rank = 1

for combo_key, count in combo_counts.most_common():
    stats = combo_stats[combo_key]
    combo_str = stats['combo_str']
    sdg_count = len(combo_key)

    # 计算频率（基于总样本数）
    frequency = count / len(df) * 100

    # 计算Layer比例
    layer_counts = stats['layer_counts']
    high_pct = (layer_counts['High'] / count * 100) if count > 0 else 0
    moderate_pct = (layer_counts['Moderate'] / count * 100) if count > 0 else 0
    low_pct = (layer_counts['Low'] / count * 100) if count > 0 else 0

    # 计算Index Score统计
    scores = stats['scores']
    if len(scores) > 0:
        score_mean = np.mean(scores)
        score_median = np.median(scores)
        score_count = len(scores)
    else:
        score_mean = score_median = np.nan
        score_count = 0

    combo_analysis_data.append({
        '排名': rank,
        'SDG组合': combo_str,
        '包含SDG数量': sdg_count,
        '出现次数': count,
        '频率(%)': round(frequency, 2),
        'High比例(%)': round(high_pct, 2),
        'Moderate比例(%)': round(moderate_pct, 2),
        'Low比例(%)': round(low_pct, 2),
        'IndexScore有效样本数': score_count,
        'IndexScore均值': round(score_mean, 2) if not np.isnan(score_mean) else 'N/A',
        'IndexScore中位数': round(score_median, 2) if not np.isnan(score_median) else 'N/A'
    })
    rank += 1

combo_analysis_df = pd.DataFrame(combo_analysis_data)

# ==================== Sheet 4: 各Layer中US的频率分布 ====================
# 先处理US数据
for col in us_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 创建US存在矩阵
us_matrix = np.zeros((len(df), len(us_columns)), dtype=int)
for idx, (_, row) in enumerate(df.iterrows()):
    for j, us_col in enumerate(us_columns):
        if pd.notna(row[us_col]) and row[us_col] == 1:
            us_matrix[idx, j] = 1

us_matrix_df = pd.DataFrame(us_matrix, columns=[us.replace('_', ' ') for us in us_columns])

# 分析：在每个Layer中，每个US出现的频率
layer_us_data = []

for layer in ['High', 'Moderate', 'Low']:
    # 筛选该Layer的样本
    layer_mask = df['SDG_layer_clean'] == layer
    layer_cases = df[layer_mask]
    layer_count = len(layer_cases)

    if layer_count > 0:
        row_data = {'Layer': layer, '样本数': layer_count}

        # 计算每个US在该Layer中的频率
        for us_col in us_matrix_df.columns:
            us_count = us_matrix_df.loc[layer_mask, us_col].sum()
            us_frequency = us_count / layer_count * 100
            row_data[us_col] = round(us_frequency, 2)

        layer_us_data.append(row_data)

layer_us_df = pd.DataFrame(layer_us_data)

# ==================== 保存结果到Excel ====================
output_file = f"{output_path}\\SDG_US_analysis.xlsx"

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Sheet 1: 单个SDG统计分析 (按SDG编号排序)
    sdg_analysis_df.to_excel(writer, sheet_name='单个SDG分析', index=False)

    # Sheet 2: SDG数量分布统计
    sdg_count_df.to_excel(writer, sheet_name='SDG数量分布', index=False)

    # Sheet 3: SDG组合统计分析 (按频率降序，带排名)
    combo_analysis_df.to_excel(writer, sheet_name='SDG组合分析', index=False)

    # Sheet 4: 各Layer中US的频率分布
    layer_us_df.to_excel(writer, sheet_name='各Layer_US频率', index=False)

print(f"分析完成！结果已保存到: {output_file}")
print(f"1. 单个SDG分析: {len(sdg_analysis_df)}行 (SDG 1-17)")
print(f"2. SDG数量分布: {len(sdg_count_df)}行 (0-{int(df['SDG_count'].max())}个SDG)")
print(f"3. SDG组合分析: {len(combo_analysis_df)}行 (前{len(combo_analysis_df)}个组合)")
print(f"4. 各Layer中US频率: {len(layer_us_df)}行 (3个Layer)")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import os

# ==================== 基本设置 ====================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 5

# 读取数据
excel_file = r"D:\CDT\04-结果\SDG分析_v3\SDG_US_analysis.xlsx"
save_path = r"D:\CDT\04-结果\SDG分析_v3"
os.makedirs(save_path, exist_ok=True)

# ==================== 1. 读取各Layer_US频率数据 ====================
layer_us_df = pd.read_excel(excel_file, sheet_name='各Layer_US频率')
print(f"各Layer_US频率数据行数: {len(layer_us_df)}")

# 提取数据
layers = layer_us_df['Layer'].tolist()
us_columns = [col for col in layer_us_df.columns if col not in ['Layer', '样本数']]

# 反转US列的顺序
us_columns_reversed = us_columns[::-1]  # 反转顺序
print(f"原始US顺序: {us_columns}")
print(f"反转后US顺序: {us_columns_reversed}")

# 创建3×6的数据矩阵（按反转顺序）
data_matrix = np.zeros((len(layers), len(us_columns_reversed)))
for i, layer in enumerate(layers):
    for j, us_col in enumerate(us_columns_reversed):
        data_matrix[i, j] = layer_us_df.loc[layer_us_df['Layer'] == layer, us_col].values[0]

print(f"数据矩阵形状: {data_matrix.shape}")
print(f"数值范围: {data_matrix.min():.1f}% 到 {data_matrix.max():.1f}%")

# ==================== 2. 创建热力图 ====================
# 创建颜色映射（从#deeeed到#68a7be）
cmap = LinearSegmentedColormap.from_list(
    'us_freq_cmap', ['#deeeed', '#68a7be']
)

# 设置颜色范围：30%到80%
vmin = 30
vmax = 80

# 创建图形（5cm宽×2.5cm高，为色带留出空间）
fig, ax = plt.subplots(figsize=(7/2.54, 3/2.54), dpi=600)
fig.subplots_adjust(left=0.05, right=0.85, bottom=0.05, top=0.95)  # 右侧留空间给色带

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# 设置单元格大小
cell_width = 1.0
cell_height = 1.0

# 绘制热力图
for i in range(len(layers)):
    for j in range(len(us_columns_reversed)):
        value = data_matrix[i, j]

        # 计算颜色（将值缩放到30-80%范围内）
        norm_value = max(0, min(1, (value - vmin) / (vmax - vmin)))
        color = cmap(norm_value)

        # 计算位置
        x = j + 0.5
        y = len(layers) - i - 0.5  # 反转y轴，使High在最上面

        # 绘制方块
        square = mpatches.Rectangle(
            (x - cell_width/2, y - cell_height/2),
            cell_width, cell_height,
            facecolor=color,
            edgecolor='white',
            linewidth=5,  # 减少线宽
            alpha=1
        )
        ax.add_patch(square)

        # 在方块中心标注百分比
        ax.text(x, y, f"{value:.0f}%",
               ha='center', va='center',
               fontsize=5, color='black',
               fontfamily='Arial')

# 设置坐标轴范围
ax.set_xlim(0, len(us_columns_reversed))
ax.set_ylim(0, len(layers))
ax.set_aspect('equal')
ax.axis('off')

# ==================== 3. 添加色带图例 ====================
# 在主图右侧添加色带
cax = fig.add_axes([0.88, 0.05, 0.03, 0.90])  # [left, bottom, width, height]

norm = plt.Normalize(vmin=vmin, vmax=vmax)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])

cbar = plt.colorbar(sm, cax=cax, orientation='vertical')
cbar.set_label('Frequency (%)', fontsize=5, fontfamily='Arial', fontweight='normal')  # 7号，不加粗
cbar.ax.tick_params(labelsize=5, width=0.25, length=2 )  # 刻度标签也用7号

# 设置色带刻度
cbar.set_ticks([30, 40, 50, 60, 70, 80])
cbar.set_ticklabels(['30%', '40%', '50%', '60%', '70%', '80%'])

cbar.outline.set_linewidth(0)  # 设置外边框线宽为0
for spine in cbar.ax.spines.values():
    spine.set_visible(False)  # 隐藏所有边框

# ==================== 4. 保存热力图 ====================
output_file = os.path.join(save_path, "Layer_US_frequency_heatmap.png")
plt.savefig(output_file, dpi=600, bbox_inches='tight', pad_inches=0.02)
plt.close()

print(f"\n热力图已保存: {output_file}")
print(f"尺寸: 5cm × 2.5cm")
print(f"网格: {len(layers)}行 × {len(us_columns_reversed)}列")
print(f"US顺序: 已反转")
print(f"颜色范围: {vmin}% (浅色) 到 {vmax}% (深色)")
print(f"实际数值范围: {data_matrix.min():.1f}% 到 {data_matrix.max():.1f}%")

# ==================== 5. 显示数据摘要（按新顺序） ====================
print(f"\n各Layer_US频率数据（反转顺序）:")
print("Layer\t" + "\t".join(us_columns_reversed))
for i, layer in enumerate(layers):
    row_values = [f"{data_matrix[i, j]:.0f}%" for j in range(len(us_columns_reversed))]
    print(f"{layer}\t" + "\t".join(row_values))

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==================== 基本设置 ====================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 5

# 读取数据
excel_file = r"D:\CDT\04-结果\SDG分析_v3\SDG_US_analysis.xlsx"
save_path = r"D:\CDT\04-结果\SDG分析_v3"

sdg_analysis_df = pd.read_excel(excel_file, sheet_name='单个SDG分析')
sdg_numbers = sdg_analysis_df['SDG编号'].values
frequencies = sdg_analysis_df['频率(%)'].values
high_pct = sdg_analysis_df['High比例(%)'].values
moderate_pct = sdg_analysis_df['Moderate比例(%)'].values
low_pct = sdg_analysis_df['Low比例(%)'].values

# 检查layer数据是否完整
has_layer_data = np.zeros(len(sdg_numbers), dtype=bool)
for i in range(len(sdg_numbers)):
    total = high_pct[i] + moderate_pct[i] + low_pct[i]
    if total > 0.01 and not np.isnan(total):
        has_layer_data[i] = True
        if abs(total - 100) > 0.01:
            high_pct[i] = high_pct[i] / total * 100
            moderate_pct[i] = moderate_pct[i] / total * 100
            low_pct[i] = low_pct[i] / total * 100

# ==================== 创建图形 ====================
fig, ax = plt.subplots(figsize=(5/2.54, 3/2.54), dpi=600)
fig.subplots_adjust(0, 0, 1, 1)

fig.patch.set_facecolor('white')
ax.set_facecolor('white')

colors = ['#f6a8a3', '#ffedd1', '#c4e4df']
missing_color = '#ececea'
bar_width = 0.6
x_positions = np.arange(len(sdg_numbers))

# 绘制条形图
for i in range(len(sdg_numbers)):
    if has_layer_data[i]:
        bottom = 0
        ax.bar(x_positions[i], low_pct[i], bar_width, color=colors[2], edgecolor='white', linewidth=0.5, bottom=bottom)
        bottom += low_pct[i]
        ax.bar(x_positions[i], moderate_pct[i], bar_width, color=colors[1], edgecolor='white', linewidth=0.5, bottom=bottom)
        bottom += moderate_pct[i]
        ax.bar(x_positions[i], high_pct[i], bar_width, color=colors[0], edgecolor='white', linewidth=0.5, bottom=bottom)
    else:
        ax.bar(x_positions[i], 100, bar_width, color=missing_color, edgecolor='white', linewidth=0.5)

# 绘制折线图
ax.plot(x_positions, frequencies, color='#68a7be', linewidth=0.5, alpha=1, zorder=5)
ax.scatter(x_positions, frequencies, color='#ee7e77', s=5, edgecolor='black', linewidth=0.3, zorder=10)
#'#ee8d87', '#fedfb2', '#A9CACE', '#feece7', '#fff9eb', '#deeeed'
# 设置坐标轴
ax.set_xlim(-0.5, len(sdg_numbers) - 0.5)
ax.set_ylim(-1, 100)
ax.axis('off')

# 保存并显示
output_file = os.path.join(save_path, "SDG_distribution_chart.png")
plt.savefig(output_file, dpi=600, bbox_inches='tight')
plt.show()
plt.close()

print(f"图表已保存: {output_file}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# ==================== 基本设置 ====================
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 5

# 读取数据
excel_file = r"D:\CDT\04-结果\SDG分析_v3\SDG_US_analysis.xlsx"
save_path = r"D:\CDT\04-结果\SDG分析_v3"

# ==================== 1. SDG数量分布条形图 (2cm×1.8cm) ====================
sdg_count_df = pd.read_excel(excel_file, sheet_name='SDG数量分布')
print(f"SDG数量分布数据行数: {len(sdg_count_df)}")

fig1, ax1 = plt.subplots(figsize=(2/2.54, 1.8/2.54), dpi=600)
fig1.subplots_adjust(0, 0, 1, 1)

fig1.patch.set_facecolor('white')
ax1.set_facecolor('white')

# 提取数据
counts = sdg_count_df['SDG数量'].values
frequencies = sdg_count_df['比例(%)'].values
samples = sdg_count_df['样本数'].values  # 样本数量

# 绘制条形图（使用#A9CACE颜色）
bars = ax1.bar(counts, frequencies, width=0.6, color='#A9CACE', edgecolor='white', linewidth=0.5)

# 在每个条形上方标注样本数量
for i, (bar, sample) in enumerate(zip(bars, samples)):
    height = bar.get_height()
    # 在条形上方标注样本数量
    ax1.text(bar.get_x() + bar.get_width()/2, height + 0.5,
             str(int(sample)),
             ha='center', va='bottom',
             fontsize=5, fontfamily='Arial')

# 设置坐标轴
ax1.set_xlim(counts.min() - 0.5, counts.max() + 0.5)
ax1.set_ylim(0, max(frequencies) * 1.2)  # 增加一点空间给标注
ax1.axis('off')

# 保存
output_file1 = os.path.join(save_path, "SDG_count_distribution.png")
plt.savefig(output_file1, dpi=600, bbox_inches='tight')
plt.close()

print(f"1. SDG数量分布条形图已保存: {output_file1}")
print(f"   尺寸: 2cm × 1.8cm")
print(f"   条形数量: {len(counts)}")
print(f"   颜色: #A9CACE")

# ==================== 2. TOP10 SDG组合横向条形图 (1.5cm×3cm) ====================
combo_analysis_df = pd.read_excel(excel_file, sheet_name='SDG组合分析')
print(f"\nSDG组合分析数据行数: {len(combo_analysis_df)}")

# 获取TOP10数据
top10_df = combo_analysis_df.head(10).copy()
top10_df = top10_df.sort_values('频率(%)', ascending=True)  # 从小到大排序，用于横向条形图

fig2, ax2 = plt.subplots(figsize=(1.5/2.54, 3/2.54), dpi=600)
fig2.subplots_adjust(0, 0, 1, 1)

fig2.patch.set_facecolor('white')
ax2.set_facecolor('white')

# 提取数据
y_positions = np.arange(len(top10_df))  # Y轴位置
frequencies_top10 = top10_df['频率(%)'].values

# 绘制横向条形图
bars2 = ax2.barh(y_positions, frequencies_top10, height=0.6, color='#ee8d87', edgecolor='white', linewidth=0.5)

# 在每个条形右侧标注频率百分比
for i, bar in enumerate(bars2):
    width = bar.get_width()
    # 在条形右侧标注百分比
    ax2.text(width + 0.5, bar.get_y() + bar.get_height()/2,
             f"{frequencies_top10[i]:.1f}%",
             ha='left', va='center',
             fontsize=5, fontfamily='Arial')

# 设置坐标轴
max_freq = max(frequencies_top10)
ax2.set_xlim(0, max_freq * 1.3)  # 增加空间给标注
ax2.set_ylim(-0.5, len(y_positions) - 0.5)
ax2.axis('off')

# 保存
output_file2 = os.path.join(save_path, "SDG_combo_top10.png")
plt.savefig(output_file2, dpi=600, bbox_inches='tight')
plt.close()

print(f"2. TOP10 SDG组合横向条形图已保存: {output_file2}")
print(f"   尺寸: 1.5cm × 3cm")
print(f"   条形数量: 10")
print(f"   颜色: #ee8d87")
print(f"   频率范围: {min(frequencies_top10):.2f}% 到 {max(frequencies_top10):.2f}%")

# ==================== 显示数据信息 ====================
print(f"\nTOP10 SDG组合详情:")
for i, (idx, row) in enumerate(top10_df.iterrows()):
    print(f"{i+1:2d}. {row['SDG组合'][:30]:30s} - {row['频率(%)']:5.2f}%")

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency
import os

# 设置路径
base_path = r"D:\CDT"
data_path = os.path.join(base_path, "03-数据", "CDT_v3.csv")
output_path = os.path.join(base_path, "04-结果", "US_SDG关联分析_v3")
os.makedirs(output_path, exist_ok=True)

# 加载数据
df = pd.read_csv(data_path)

# 数据清洗
us_columns = ['People', 'Household', 'Land_and_agriculture',
              'Workplace', 'Transportation', 'IoT_technology']
us_columns = [col for col in us_columns if col in df.columns]

for col in us_columns:
    df[col] = pd.to_numeric(df[col], errors='coerce')
    df[col] = np.where(df[col] == 1, 1, 0)

def parse_sdgs(cell):
    if pd.isna(cell):
        return []
    cell_str = str(cell).strip()
    if not cell_str:
        return []

    parts = cell_str.replace(', ', ',').split(',')
    sdg_list = []
    for part in parts:
        part_clean = part.strip()
        if part_clean:
            try:
                sdg_num = int(part_clean)
                if 1 <= sdg_num <= 17:
                    sdg_list.append(sdg_num)
            except:
                pass
    return sorted(list(set(sdg_list)))

df['SDG_list'] = df['Covered_SDGs'].apply(parse_sdgs)
df['SDG_layer'] = df['2025 SDG layer'].map({'High': 'High', 'Moderate': 'Moderate', 'Low': 'Low'})

# 创建SDG存在矩阵
sdg_matrix = np.zeros((len(df), 17), dtype=int)
for idx, sdg_list in enumerate(df['SDG_list']):
    for sdg in sdg_list:
        if 1 <= sdg <= 17:
            sdg_matrix[idx, sdg-1] = 1

# 关联分析函数
def calculate_association(us_series, sdg_series, min_cooccur=1):
    cooccur = ((us_series == 1) & (sdg_series == 1)).sum()
    if cooccur < min_cooccur:
        return np.nan, np.nan, np.nan, cooccur

    a = cooccur
    b = ((us_series == 1) & (sdg_series == 0)).sum()
    c = ((us_series == 0) & (sdg_series == 1)).sum()
    d = ((us_series == 0) & (sdg_series == 0)).sum()

    denominator = np.sqrt((a + b) * (c + d) * (a + c) * (b + d))
    phi = (a * d - b * c) / denominator if denominator > 0 else 0.0

    try:
        chi2, p, dof, expected = chi2_contingency([[a, b], [c, d]], correction=False)
    except:
        p = 1.0

    cond_prob = (a / (a + b) * 100) if (a + b) > 0 else 0.0
    return phi, p, cond_prob, cooccur

# 生成关联矩阵
MIN_COOCCUR = 1
output_file = os.path.join(output_path, "US_SDG_association.xlsx")

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # 整体分析
    phi_overall = pd.DataFrame(index=[col.replace('_', ' ') for col in us_columns],
                              columns=[f'SDG_{i}' for i in range(1, 18)])
    p_overall = pd.DataFrame(index=[col.replace('_', ' ') for col in us_columns],
                            columns=[f'SDG_{i}' for i in range(1, 18)])
    cond_overall = pd.DataFrame(index=[col.replace('_', ' ') for col in us_columns],
                               columns=[f'SDG_{i}' for i in range(1, 18)])

    for us_col in us_columns:
        us_name = us_col.replace('_', ' ')
        us_data = df[us_col]
        for sdg_num in range(1, 18):
            sdg_data = pd.Series(sdg_matrix[:, sdg_num-1])
            phi, p, cond, cooccur = calculate_association(us_data, sdg_data, MIN_COOCCUR)

            if cooccur >= MIN_COOCCUR:
                phi_overall.loc[us_name, f'SDG_{sdg_num}'] = round(phi, 3)
                p_overall.loc[us_name, f'SDG_{sdg_num}'] = round(p, 4)
                cond_overall.loc[us_name, f'SDG_{sdg_num}'] = round(cond, 1)

    phi_overall.to_excel(writer, sheet_name='Overall_Phi')
    p_overall.to_excel(writer, sheet_name='Overall_Pvalue')
    cond_overall.to_excel(writer, sheet_name='Overall_CondProb')

    # 分层分析
    for layer in ['High', 'Moderate', 'Low']:
        layer_mask = df['SDG_layer'] == layer
        if layer_mask.sum() > 0:
            layer_df = df[layer_mask]
            layer_sdg = sdg_matrix[layer_mask]

            phi_layer = pd.DataFrame(index=[col.replace('_', ' ') for col in us_columns],
                                    columns=[f'SDG_{i}' for i in range(1, 18)])
            p_layer = pd.DataFrame(index=[col.replace('_', ' ') for col in us_columns],
                                  columns=[f'SDG_{i}' for i in range(1, 18)])
            cond_layer = pd.DataFrame(index=[col.replace('_', ' ') for col in us_columns],
                                     columns=[f'SDG_{i}' for i in range(1, 18)])

            for us_col in us_columns:
                us_name = us_col.replace('_', ' ')
                us_data = layer_df[us_col]
                for sdg_num in range(1, 18):
                    sdg_data = pd.Series(layer_sdg[:, sdg_num-1])
                    phi, p, cond, cooccur = calculate_association(us_data, sdg_data, MIN_COOCCUR)

                    if cooccur >= MIN_COOCCUR:
                        phi_layer.loc[us_name, f'SDG_{sdg_num}'] = round(phi, 3)
                        p_layer.loc[us_name, f'SDG_{sdg_num}'] = round(p, 4)
                        cond_layer.loc[us_name, f'SDG_{sdg_num}'] = round(cond, 1)

            phi_layer.to_excel(writer, sheet_name=f'{layer}_Phi')
            p_layer.to_excel(writer, sheet_name=f'{layer}_Pvalue')
            cond_layer.to_excel(writer, sheet_name=f'{layer}_CondProb')

print(f"完成: {output_file}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import os

# ==================== 基本设置 ====================
type = 'Overall'
plt.rcParams['font.family'] = 'Arial'
plt.rcParams['font.size'] = 5
plt.rcParams['font.weight'] = 'normal'

# 路径设置
save_path = r"D:\CDT\04-结果\fig3_v3"
excel_file = r"D:\CDT\04-结果\US_SDG关联分析\US_SDG_association.xlsx"

# ==================== 1. 从Excel读取数据 ====================
print("读取Excel文件...")
phi_df = pd.read_excel(excel_file, sheet_name=f'{type}_Phi', index_col=0)
p_df = pd.read_excel(excel_file, sheet_name=f'{type}_Pvalue', index_col=0)
cond_df = pd.read_excel(excel_file, sheet_name=f'{type}_CondProb', index_col=0)

# 转换数据为numpy数组
phi_matrix = phi_df.replace('', np.nan).astype(float).values
p_matrix = p_df.replace('', np.nan).astype(float).values
cond_matrix = cond_df.replace('', np.nan).astype(float).values

# 获取US名称和SDG名称，处理SDG名称
us_names = phi_df.index.tolist()
sdg_names = [name.replace('_', ' ') for name in phi_df.columns.tolist()]  # 去掉下划线
n_us = len(us_names)
n_sdg = len(sdg_names)

print(f"US数量: {n_us}, SDG数量: {n_sdg}")

# ==================== 2. 设置颜色映射 ====================
# 获取有效的phi值用于归一化
valid_phi = phi_matrix[~np.isnan(phi_matrix)]
if len(valid_phi) > 0:
    vmax = max(abs(valid_phi.min()), abs(valid_phi.max()))
    vmin = -vmax
else:
    vmax = 1.0
    vmin = -1.0

# 创建颜色映射
cmap = LinearSegmentedColormap.from_list(
    'phi_cmap', ['#68a7be', '#ffffff', '#ee7e77']
)

# ==================== 3. 创建8cm×5cm热图 ====================
fig, ax = plt.subplots(figsize=(9/2.54, 5.5/2.54), dpi=600)
fig.subplots_adjust(left=0.05, right=0.95, bottom=0.05, top=0.95)

# 方块大小参数
min_size = 0.5   # 最小方块大小（对应条件概率0%）
max_size = 1.0   # 最大方块大小（对应条件概率100%）

# ==================== 3.1 绘制热图 ====================
for i in range(n_us):
    for j in range(n_sdg):
        phi_val = phi_matrix[i, j]
        p_val = p_matrix[i, j]
        cond_val = cond_matrix[i, j]

        # 跳过NaN值
        if np.isnan(phi_val) or np.isnan(cond_val):
            continue

        # 计算颜色
        norm_val = (phi_val - vmin) / (vmax - vmin)
        color = cmap(norm_val)

        # 计算方块大小（基于条件概率）
        cell_size = min_size + (cond_val / 100) * (max_size - min_size)

        # 计算位置
        x = j + 0.5  # 居中
        y = n_us - i - 0.5  # 反转y轴

        # 绘制方块
        square = mpatches.Rectangle(
            (x - cell_size/2, y - cell_size/2),
            cell_size, cell_size,
            facecolor=color,
            edgecolor='white',
            linewidth=0.3,
            alpha=0.9
        )
        ax.add_patch(square)

        # 检查显著性
        if not np.isnan(p_val) and p_val < 0.05:
            # 显著性星号
            if p_val < 0.001:
                sig = "***"
            elif p_val < 0.01:
                sig = "**"
            else:
                sig = "*"

            # 根据背景色选择文字颜色
            if abs(phi_val) > 0.4:
                text_color = 'white'
            else:
                text_color = 'black'

            # 系数居中显示（上方）
            ax.text(x, y - 0.06, f"{phi_val:.2f}",
                   ha='center', va='center',
                   fontsize=5, color=text_color)

            # 星号居中显示（下方）
            ax.text(x, y + 0.33, sig,
                   ha='center', va='center',
                   fontsize=5, color=text_color)

# 设置坐标轴范围
ax.set_xlim(0, n_sdg)
ax.set_ylim(0, n_us)
ax.set_aspect('equal')
ax.invert_yaxis()
ax.axis('off')  # 完全关闭坐标轴

# ==================== 4. 保存并显示主图 ====================
output_file = os.path.join(save_path, f"fig3g_{type}.png")
plt.savefig(output_file, dpi=600, bbox_inches='tight', pad_inches=0.02)
print(f"主图已保存: {output_file}")

# 显示图形
plt.show()
plt.close()

# ==================== 5. 创建单独的颜色条图例 ====================
fig_c, ax_c = plt.subplots(figsize=(0.6/2.54, 3/2.54), dpi=600)
fig_c.subplots_adjust(left=0.3, right=0.7, top=0.98, bottom=0.02)

sm = plt.cm.ScalarMappable(
    cmap=cmap,
    norm=plt.Normalize(vmin=vmin, vmax=vmax)
)
sm.set_array([])

cbar = fig_c.colorbar(sm, cax=ax_c, orientation='vertical')
cbar.set_label('Phi', fontsize=5, labelpad=2)
cbar.ax.tick_params(labelsize=5, width=0.25, length=2)

for spine in ax_c.spines.values():
    spine.set_linewidth(0)

colorbar_file = os.path.join(save_path, f"fig3g_legend_{type}.png")
plt.savefig(colorbar_file, dpi=600, bbox_inches='tight')
plt.close()

print(f"颜色条图例已保存: {colorbar_file}")